# Multi-Agent Travel Planner (LangGraph)

**Mini-Project Sprint — Prompt Engineering & Autonomous Reasoning**

One travel request goes in. A team of agents works on it. A refined travel plan comes out.

### The team

| Who | Job | Tools it may use |
|---|---|---|
| **Manager** | Understands the request, decides who works, writes the plan | none |
| **Flight Agent** | Finds and ranks flights | flights, currency |
| **Hotel Agent** | Finds and ranks hotels | hotels, currency, tips |
| **Activity Agent** | Plans day-wise things to do | activities, restaurants, weather |
| **Critic Agent** | Checks the plan for mistakes | none |

### The flow

```
User request
     |
  MANAGER  (reads the request, picks the workers)
     |
     +--------------+--------------+      <- these three run at the SAME TIME
  FLIGHT         HOTEL         ACTIVITY
     +--------------+--------------+
     |
  SYNTHESIS  (manager writes the first plan)
     |
  CRITIC     (finds the problems)
     |
  REVISION   (manager fixes only those problems)
     |
  Final travel plan
```

### How to run this notebook

Run every cell from top to bottom. The last cell is where you type your own trip request.


## Step 1 — Install the libraries

Three packages do all the heavy lifting for us.

- `langchain` — gives us tools and agents
- `langgraph` — connects the agents into a flow
- `langchain-openai` — talks to the OpenAI model

Run this cell once. It takes about a minute.


In [ ]:
!pip install -q langchain langgraph langchain-openai
print("Libraries installed.")

## Step 2 — Add your OpenAI key

Two ways. The notebook tries them in order.

**Best way — Colab Secrets.** Click the **key icon** in the left sidebar of Colab.
Add a secret named `OPENAI_API_KEY`, paste your key, and turn on **Notebook access**.
Do this once and you never type the key again.

**Fallback.** If no secret is found, a hidden box appears and you paste the key there.


In [ ]:
import os

# The key is read from Colab Secrets: the key icon in the left sidebar.
# The secret must be named exactly OPENAI_API_KEY and "Notebook access" must be ON.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Key loaded from Colab Secrets.")
except Exception as problem:
    # Print the real reason before falling back, so you know what to fix.
    print("Could not read the secret. Reason:", type(problem).__name__, "-", problem)
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key and press Enter: ")
    print("Key accepted.")

## Step 3 — The travel data

A real travel website would call an API here. We use plain Python lists instead.

Why? Because a travel API costs money, needs approval, and breaks during a demo.
The agents cannot tell the difference — they ask a question and get data back either way.

Later you can swap these lists for a real API and change nothing else.


In [ ]:
# ---------- FLIGHTS ----------
FLIGHTS = [
    {"id": "SQ-421", "from": "Mumbai", "to": "Singapore", "airline": "Singapore Airlines",
     "depart": "11:45", "arrive": "19:50", "stops": 0, "price_inr": 28500, "red_eye": False},
    {"id": "AI-342", "from": "Mumbai", "to": "Singapore", "airline": "Air India",
     "depart": "23:15", "arrive": "07:20", "stops": 0, "price_inr": 23600, "red_eye": True},
    {"id": "6E-1071", "from": "Mumbai", "to": "Singapore", "airline": "IndiGo",
     "depart": "08:30", "arrive": "16:40", "stops": 0, "price_inr": 21900, "red_eye": False},
    {"id": "MH-199", "from": "Mumbai", "to": "Singapore", "airline": "Malaysia Airlines",
     "depart": "14:20", "arrive": "23:55", "stops": 1, "price_inr": 19400, "red_eye": False},
    {"id": "SQ-423", "from": "Delhi", "to": "Singapore", "airline": "Singapore Airlines",
     "depart": "10:10", "arrive": "19:05", "stops": 0, "price_inr": 31200, "red_eye": False},
    {"id": "6E-1085", "from": "Bangalore", "to": "Singapore", "airline": "IndiGo",
     "depart": "09:05", "arrive": "16:20", "stops": 0, "price_inr": 20800, "red_eye": False},
]

# ---------- HOTELS ----------
HOTELS = [
    {"name": "Hotel Boss", "city": "Singapore", "area": "Lavender",
     "price_per_night_inr": 7200, "rating": 4.0, "style": "budget"},
    {"name": "Village Hotel Bugis", "city": "Singapore", "area": "Bugis",
     "price_per_night_inr": 9800, "rating": 4.3, "style": "mid"},
    {"name": "Hotel G Singapore", "city": "Singapore", "area": "Bencoolen",
     "price_per_night_inr": 8600, "rating": 4.2, "style": "mid"},
    {"name": "Marina Bay Sands", "city": "Singapore", "area": "Marina Bay",
     "price_per_night_inr": 34000, "rating": 4.7, "style": "luxury"},
    {"name": "Hotel Mono", "city": "Singapore", "area": "Chinatown",
     "price_per_night_inr": 10500, "rating": 4.4, "style": "mid"},
    {"name": "The Sultan", "city": "Singapore", "area": "Kampong Glam",
     "price_per_night_inr": 11200, "rating": 4.5, "style": "mid"},
]

# ---------- THINGS TO DO ----------
ACTIVITIES = [
    {"name": "Gardens by the Bay", "city": "Singapore", "interest": "city views", "hours": 3, "area": "Marina Bay"},
    {"name": "Marina Bay Sands SkyPark", "city": "Singapore", "interest": "city views", "hours": 2, "area": "Marina Bay"},
    {"name": "Singapore Flyer", "city": "Singapore", "interest": "city views", "hours": 2, "area": "Marina Bay"},
    {"name": "Chinatown Heritage Centre", "city": "Singapore", "interest": "culture", "hours": 2, "area": "Chinatown"},
    {"name": "Sultan Mosque walk", "city": "Singapore", "interest": "culture", "hours": 2, "area": "Kampong Glam"},
    {"name": "Little India walking tour", "city": "Singapore", "interest": "culture", "hours": 3, "area": "Little India"},
    {"name": "Hawker food trail", "city": "Singapore", "interest": "food", "hours": 3, "area": "Chinatown"},
    {"name": "Sentosa beach day", "city": "Singapore", "interest": "relax", "hours": 5, "area": "Sentosa"},
]

# ---------- FOOD ----------
RESTAURANTS = [
    {"name": "Maxwell Food Centre", "city": "Singapore", "cuisine": "local", "cost_inr": 500, "area": "Chinatown"},
    {"name": "Lau Pa Sat", "city": "Singapore", "cuisine": "local", "cost_inr": 600, "area": "Downtown"},
    {"name": "Komala Vilas", "city": "Singapore", "cuisine": "indian veg", "cost_inr": 700, "area": "Little India"},
    {"name": "Zam Zam", "city": "Singapore", "cuisine": "halal", "cost_inr": 800, "area": "Kampong Glam"},
    {"name": "Newton Food Centre", "city": "Singapore", "cuisine": "seafood", "cost_inr": 1600, "area": "Newton"},
    {"name": "Tiong Bahru Market", "city": "Singapore", "cuisine": "local", "cost_inr": 450, "area": "Tiong Bahru"},
]

# ---------- WEATHER ----------
WEATHER = {
    "singapore": "Hot and humid all year, 26-32 C. Short heavy rain most afternoons. Carry an umbrella.",
    "bangkok":   "Hot, 28-35 C. Heavy rain June to October.",
    "dubai":     "Very hot April to September, 35-45 C. Pleasant November to March.",
}

# ---------- LOCAL TIPS ----------
TIPS = {
    "singapore": "Indian passport holders need a visa. MRT covers the whole city. Buy an EZ-Link card. Tap water is safe. No chewing gum.",
    "bangkok":   "Visa on arrival for Indians. Use BTS Skytrain. Always agree the taxi fare first.",
    "dubai":     "Visa needed in advance. Metro is clean and cheap. Dress modestly in public places.",
}

# ---------- EXCHANGE RATES (1 unit = ? INR) ----------
RATES = {"SGD": 65.0, "USD": 88.0, "THB": 2.5, "AED": 24.0, "INR": 1.0}

print("Data loaded:", len(FLIGHTS), "flights,", len(HOTELS), "hotels,",
      len(ACTIVITIES), "activities,", len(RESTAURANTS), "restaurants.")

## Step 4 — The tools

A **tool** is a normal Python function that an agent is allowed to call.

Two rules make a tool work:

1. Put `@tool` on top of the function.
2. Write a clear one-line description inside `""" """`.

That description is the only thing the agent reads when deciding whether to use the tool.
A vague description means the agent picks the wrong tool. This is the most common bug in agent projects.

We build **seven** tools.


In [ ]:
from langchain.tools import tool


@tool
def search_flights(origin: str, destination: str, avoid_red_eye: bool = False,
                   max_price_inr: int = 999999) -> list:
    """Find flights between two cities. Set avoid_red_eye to True to remove overnight flights."""
    found = [f for f in FLIGHTS
             if f["from"].lower() == origin.lower()
             and f["to"].lower() == destination.lower()
             and f["price_inr"] <= max_price_inr]
    if avoid_red_eye:
        found = [f for f in found if not f["red_eye"]]
    return sorted(found, key=lambda f: f["price_inr"])


@tool
def search_hotels(city: str, max_price_per_night_inr: int = 999999, style: str = "any") -> list:
    """Find hotels in a city. Style can be budget, mid, luxury or any."""
    found = [h for h in HOTELS
             if h["city"].lower() == city.lower()
             and h["price_per_night_inr"] <= max_price_per_night_inr]
    if style != "any":
        found = [h for h in found if h["style"] == style.lower()]
    return sorted(found, key=lambda h: -h["rating"])


@tool
def search_activities(city: str, interest: str = "any") -> list:
    """Find things to do in a city. Interest can be food, culture, city views, relax or any."""
    found = [a for a in ACTIVITIES if a["city"].lower() == city.lower()]
    if interest != "any":
        found = [a for a in found if interest.lower() in a["interest"]]
    return found


@tool
def search_restaurants(city: str, cuisine: str = "any") -> list:
    """Find places to eat in a city. Cuisine can be local, indian veg, halal, seafood or any."""
    found = [r for r in RESTAURANTS if r["city"].lower() == city.lower()]
    if cuisine != "any":
        found = [r for r in found if cuisine.lower() in r["cuisine"]]
    return sorted(found, key=lambda r: r["cost_inr"])


@tool
def get_weather(city: str) -> str:
    """Get the usual weather for a city, so the plan can allow for rain or heat."""
    return WEATHER.get(city.lower(), "No weather information for this city.")


@tool
def get_travel_tips(city: str) -> str:
    """Get visa, transport and local rules for a city."""
    return TIPS.get(city.lower(), "No tips available for this city.")


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies, for example SGD to INR."""
    if from_currency.upper() not in RATES or to_currency.upper() not in RATES:
        return "Currency not supported."
    in_inr = amount * RATES[from_currency.upper()]
    result = in_inr / RATES[to_currency.upper()]
    return f"{amount} {from_currency.upper()} = {round(result, 2)} {to_currency.upper()}"


print("7 tools ready.")

### Try one tool on its own

Before giving a tool to an agent, always check it works by itself.
If the tool is broken, the agent looks broken, and you will waste an hour blaming the prompt.


In [ ]:
# .invoke() runs a tool directly. The input is a dictionary of its arguments.
search_flights.invoke({"origin": "Mumbai", "destination": "Singapore", "avoid_red_eye": True})

## Step 5 — The tool registry

We now have seven tools. Nobody gets all seven.

A **registry** is a simple dictionary that says which agent gets which tools.

Why bother? Because an agent with too many tools starts guessing. The hotel agent has
no business looking at flights. Fewer tools means better choices and easier debugging.

This is the same rule from the slides: *if two agents can change the same decision, the design will drift.*


In [ ]:
TOOL_REGISTRY = {
    "flight_agent":   [search_flights, convert_currency],
    "hotel_agent":    [search_hotels, convert_currency, get_travel_tips],
    "activity_agent": [search_activities, search_restaurants, get_weather],
    "critic_agent":   [],   # the critic only reads the plan, it does not search
}

# Show the registry as a small table
for agent_name, tools in TOOL_REGISTRY.items():
    tool_names = [t.name for t in tools] or ["(no tools)"]
    print(f"{agent_name:16} -> {', '.join(tool_names)}")

## Step 6 — The agents

`create_agent` builds a complete ReAct agent in one line. ReAct means the agent loops through:

**Reason** (what do I need?) → **Act** (call a tool) → **Observe** (read the result) → **Decide** (answer)

We do not write that loop. LangChain does it.

Each agent gets three things: a model, its tools from the registry, and a **contract** —
a system prompt saying exactly what to return. Same contract every time means the
manager can join the answers together without surprises.


In [ ]:
from langchain.agents import create_agent

MODEL = "openai:gpt-4o-mini"     # small, fast and cheap. Change to gpt-4o for better writing.


def ask(agent, question: str) -> str:
    """Send a question to an agent and get its final answer back as plain text."""
    reply = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return reply["messages"][-1].content


# ---------- The manager. No tools. It thinks and writes. ----------
manager_agent = create_agent(
    model=MODEL,
    tools=[],
    system_prompt=(
        "You are the manager of a travel planning team. "
        "You never search for travel data yourself. "
        "You read the request, decide who should work, and write the final plan. "
        "Be short and clear. Never invent a flight, hotel or place that a worker did not give you."
    ),
)

# ---------- Flight agent ----------
flight_agent = create_agent(
    model=MODEL,
    tools=TOOL_REGISTRY["flight_agent"],
    system_prompt=(
        "You are the flight specialist. Use your tools to find real options. "
        "Never suggest a flight that did not come from a tool. "
        "Answer in exactly this shape:\n"
        "TOP 3 OPTIONS: a numbered list with airline, flight id, times and price\n"
        "RECOMMENDED: one option and one line saying why\n"
        "TRADE-OFFS: what the traveller gives up by taking it\n"
        "ASSUMPTIONS: anything you had to guess"
    ),
)

# ---------- Hotel agent ----------
hotel_agent = create_agent(
    model=MODEL,
    tools=TOOL_REGISTRY["hotel_agent"],
    system_prompt=(
        "You are the hotel specialist. Use your tools to find real options. "
        "Never suggest a hotel that did not come from a tool. "
        "Always check the total cost for the whole stay, not only the price per night. "
        "Answer in exactly this shape:\n"
        "TOP 3 OPTIONS: a numbered list with name, area, rating, price per night, total for the stay\n"
        "RECOMMENDED: one option and one line saying why\n"
        "TRADE-OFFS: what the traveller gives up by taking it\n"
        "ASSUMPTIONS: anything you had to guess"
    ),
)

# ---------- Activity agent ----------
activity_agent = create_agent(
    model=MODEL,
    tools=TOOL_REGISTRY["activity_agent"],
    system_prompt=(
        "You are the activities specialist. Use your tools to find real places. "
        "Never suggest a place that did not come from a tool. "
        "Do not overload a day. Two or three activities plus one meal is enough. "
        "Answer in exactly this shape:\n"
        "DAY BY DAY: for each day, morning, afternoon, evening and where to eat\n"
        "WEATHER NOTE: one line\n"
        "ASSUMPTIONS: anything you had to guess"
    ),
)

# ---------- Critic agent ----------
critic_agent = create_agent(
    model=MODEL,
    tools=TOOL_REGISTRY["critic_agent"],
    system_prompt=(
        "You are a strict reviewer of travel plans. You do not rewrite the plan. "
        "You list problems only. "
        "Check four things: budget respected, days not overloaded, the traveller's stated "
        "interests actually appear, and guesses are stated honestly. "
        "Answer as a numbered list of specific fixes. "
        "Never write vague comments like 'make it better'. "
        "If the plan is fine, say NO ISSUES FOUND."
    ),
)

print("5 agents ready: manager, flight, hotel, activity, critic.")

## Step 7 — The shared state

Every agent in the flow writes into one shared box called the **state**.

Notice each agent has its **own key** — `flight_report`, `hotel_report`, `activity_report`.
This matters, because the three workers run at the same time. If two of them wrote into the
same key at the same moment, LangGraph would stop with an error.

Separate keys, no clash, no extra code.


In [ ]:
from typing import TypedDict, List


class TravelState(TypedDict):
    request: str            # what the user typed
    requirements: str       # what the manager understood
    chosen_workers: List[str]  # who the manager picked
    flight_report: str      # written by the flight agent
    hotel_report: str       # written by the hotel agent
    activity_report: str    # written by the activity agent
    draft_plan: str         # the manager's first attempt
    critique: str           # the critic's list of problems
    final_plan: str         # the fixed plan


print("State defined.")

## Step 8 — The nodes

A **node** is one step in the flow. Each node is a small function:
it reads the state, does one job, and returns what it wants to add.

Seven nodes, each only a few lines long.


In [ ]:
def manager_node(state):
    """Read the request. Write down the requirements. Decide who should work."""
    requirements = ask(manager_agent, f"""
Read this travel request and list what you understood.

REQUEST: {state['request']}

List these lines only:
FROM, TO, DAYS, TRAVELLERS, BUDGET, INTERESTS, HARD RULES, MISSING INFO
""")

    choice = ask(manager_agent, f"""
Here are the requirements:
{requirements}

Which specialists are needed? Choose from: flight_agent, hotel_agent, activity_agent.
Reply with the names separated by commas and nothing else.
""")

    # Keep only names we actually have. This protects us if the model adds something extra.
    valid = ["flight_agent", "hotel_agent", "activity_agent"]
    workers = [name for name in valid if name in choice.lower()]
    if not workers:
        workers = valid          # if unsure, use everyone

    print("MANAGER understood the request.")
    print("MANAGER picked:", ", ".join(workers))
    return {"requirements": requirements, "chosen_workers": workers}


def flight_node(state):
    """The flight agent works on its part."""
    report = ask(flight_agent, f"Find flights for this trip.\n\n{state['requirements']}")
    return {"flight_report": report}


def hotel_node(state):
    """The hotel agent works on its part."""
    report = ask(hotel_agent, f"Find hotels for this trip.\n\n{state['requirements']}")
    return {"hotel_report": report}


def activity_node(state):
    """The activity agent works on its part."""
    report = ask(activity_agent, f"Plan the days for this trip.\n\n{state['requirements']}")
    return {"activity_report": report}


def synthesis_node(state):
    """The manager joins the three reports into one travel plan."""
    draft = ask(manager_agent, f"""
Write the first version of the travel plan using ONLY the reports below.

REQUIREMENTS:
{state['requirements']}

FLIGHT REPORT:
{state.get('flight_report', 'not requested')}

HOTEL REPORT:
{state.get('hotel_report', 'not requested')}

ACTIVITY REPORT:
{state.get('activity_report', 'not requested')}

Use these headings:
CHOSEN FLIGHT, CHOSEN HOTEL, DAY BY DAY PLAN, TOTAL COST, WHY THESE CHOICES, ASSUMPTIONS
""")
    print("MANAGER wrote the first plan.")
    return {"draft_plan": draft}


def critic_node(state):
    """The critic looks for problems. It does not fix them."""
    issues = ask(critic_agent, f"""
Review this plan against the requirements.

REQUIREMENTS:
{state['requirements']}

PLAN:
{state['draft_plan']}
""")
    print("CRITIC finished the review.")
    return {"critique": issues}


def revision_node(state):
    """The manager fixes only what the critic listed. Everything else stays as it was."""
    if "NO ISSUES FOUND" in state["critique"].upper():
        print("No changes needed.")
        return {"final_plan": state["draft_plan"]}

    fixed = ask(manager_agent, f"""
Fix ONLY the problems listed below. Do not rewrite the parts that are fine.

PLAN:
{state['draft_plan']}

PROBLEMS TO FIX:
{state['critique']}

Return the full corrected plan, then a short section called WHAT CHANGED.
""")
    print("MANAGER fixed the plan.")
    return {"final_plan": fixed}


print("7 nodes ready.")

## Step 9 — Connect the nodes into a graph

Now we wire the steps together.

The interesting line is `add_conditional_edges`. The manager returns a **list** of worker names,
and LangGraph starts all of them at the same time. That is parallel work with no threading code.

Why parallel? Because the hotel agent does not need the flight answer, and the activity agent
does not need the hotel answer. They only meet again at synthesis.


In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TravelState)

# Add every step
graph.add_node("manager", manager_node)
graph.add_node("flight_agent", flight_node)
graph.add_node("hotel_agent", hotel_node)
graph.add_node("activity_agent", activity_node)
graph.add_node("synthesis", synthesis_node)
graph.add_node("critic", critic_node)
graph.add_node("revision", revision_node)

# Start at the manager
graph.add_edge(START, "manager")

# The manager's list of workers decides who runs. All of them start together.
graph.add_conditional_edges(
    "manager",
    lambda state: state["chosen_workers"],
    ["flight_agent", "hotel_agent", "activity_agent"],
)

# Whoever ran, they all meet at synthesis
graph.add_edge("flight_agent", "synthesis")
graph.add_edge("hotel_agent", "synthesis")
graph.add_edge("activity_agent", "synthesis")

# Then the quality loop
graph.add_edge("synthesis", "critic")
graph.add_edge("critic", "revision")
graph.add_edge("revision", END)

travel_planner = graph.compile()
print("Graph built.")

### See the graph

This draws the flow we just built. Useful for checking you wired it the way you meant to.


In [ ]:
from IPython.display import Image, display

try:
    display(Image(travel_planner.get_graph().draw_mermaid_png()))
except Exception:
    # If the picture cannot be drawn, print the text version instead
    print(travel_planner.get_graph().draw_mermaid())

## Step 10 — One function to run the whole thing

This prints each part of the run so you can see who did what. That is the whole point of the
project: not just an answer, but a **trace** you can inspect.


In [ ]:
def plan_trip(request: str, show_details: bool = True):
    """Run the full agent team on a travel request."""
    print("=" * 70)
    print("REQUEST:", request)
    print("=" * 70)

    result = travel_planner.invoke({"request": request})

    if show_details:
        print("\n" + "-" * 70)
        print("WHAT THE MANAGER UNDERSTOOD")
        print("-" * 70)
        print(result["requirements"])

        for key, title in [("flight_report", "FLIGHT AGENT"),
                           ("hotel_report", "HOTEL AGENT"),
                           ("activity_report", "ACTIVITY AGENT")]:
            if result.get(key):
                print("\n" + "-" * 70)
                print(title)
                print("-" * 70)
                print(result[key])

        print("\n" + "-" * 70)
        print("FIRST DRAFT")
        print("-" * 70)
        print(result["draft_plan"])

        print("\n" + "-" * 70)
        print("WHAT THE CRITIC FOUND")
        print("-" * 70)
        print(result["critique"])

    print("\n" + "=" * 70)
    print("FINAL TRAVEL PLAN")
    print("=" * 70)
    print(result["final_plan"])
    return result


print("Ready to run.")

## Step 11 — The demo run

This is the request from the slides. Watch the order of the printed lines:
the manager speaks first, the three workers finish, then the draft, then the critic, then the fix.


In [ ]:
demo_request = """Plan a 4-day Singapore trip from Mumbai for two people.
Mid-budget. Avoid red-eye flights.
Prefer food, city views and cultural sites.
Keep hotel budget under 45000 rupees."""

result = plan_trip(demo_request)

## Step 12 — Did the demo actually work?

A plan that reads nicely is not proof of anything. Check these five points against the output above.

| # | Check | What good looks like |
|---|---|---|
| 1 | **Constraint capture** | From, to, days, people, budget and hard rules are all in the requirements |
| 2 | **Tool grounding** | Every flight and hotel in the plan appears in our data. Nothing invented |
| 3 | **Trade-off reasoning** | The plan says why this option and not the cheaper one |
| 4 | **Critique quality** | The critic listed specific fixes, not "make it better" |
| 5 | **Revision trace** | The final plan is different from the draft, and says what changed |

Run the cell below to check point 2 automatically.


In [ ]:
# Check that every flight id in the final plan really exists in our data
plan_text = result["final_plan"]

real_ids = [f["id"] for f in FLIGHTS]
found_ids = [fid for fid in real_ids if fid in plan_text]
print("Real flight ids used in the plan:", found_ids or "none")

real_hotels = [h["name"] for h in HOTELS]
found_hotels = [name for name in real_hotels if name in plan_text]
print("Real hotel names used in the plan:", found_hotels or "none")

print("\nIf both lines are empty, the plan was invented and the grounding check has failed.")

## Step 13 — When it goes wrong

Agent projects fail in the same few ways. Fix them in this order — **left to right along the flow**.
Never start by changing the last prompt.

| What you see | Where the problem really is | What to do |
|---|---|---|
| A user rule was ignored | The manager's requirements | Read the requirements print. If the rule is missing there, the rest never had a chance |
| An agent gave a wall of prose | The system prompt contract | Make the required headings stricter |
| A hotel or flight was invented | The tool was never called | Check the tool description. A vague description means the agent skips the tool |
| The critic says "looks good" | The critic prompt | Force a numbered list of specific fixes |
| The whole plan got rewritten | The revision prompt | Say clearly: fix only the listed problems |
| A parallel error about a key | Two nodes writing the same state key | Give each agent its own key |

**The habit to build:** print the middle steps. Most people debug the final answer and waste an hour.
The mistake almost always happened earlier.


## Step 14 — Your turn: type any trip request

Run the cell below and type your request when the box appears.

Things to try:

- `Plan a 3-day Singapore trip from Bangalore for one person. Only food and culture. Cheapest flight.`
- `I only need flights from Delhi to Singapore under 32000 rupees.` — watch the manager pick fewer workers
- `Plan a luxury 2-day Singapore trip from Mumbai. Money is not a problem.`


In [ ]:
your_request = input("Describe your trip: ")

result = plan_trip(your_request)